# Notebook 4: Question Answering

In this notebook we explore two flavours of **question answering (QA)**:

1. **Extractive QA** – the model identifies a span in a provided context that answers the question (e.g., DistilBERT fine-tuned on SQuAD).
2. **Generative QA** – the model generates a free-form answer, optionally conditioned on retrieved context.

## Learning Objectives
- Understand the difference between extractive and generative QA
- Use the `question-answering` pipeline from Hugging Face
- Load and evaluate models fine-tuned on SQuAD
- Build a simple open-domain QA demo
- Evaluate with Exact Match (EM) and F1 metrics

## 1. Install & Import Dependencies

In [ ]:
# !pip install transformers torch datasets evaluate

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import torch
from transformers import (
    pipeline,
    AutoTokenizer,
    AutoModelForQuestionAnswering,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Extractive QA with the Pipeline API

Given a **context** (a paragraph) and a **question**, the model extracts the answer span directly from the context.

In [ ]:
qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad",
    device=0 if torch.cuda.is_available() else -1,
)
print("QA pipeline loaded.")

In [ ]:
context = """
The Hugging Face Transformers library was originally created by Thomas Wolf, Lysandre Debut,
Victor Sanh, Julien Chaumond, Clement Delangue, Anthony Moi, Pierric Cistac, and Tim Rault.
It provides thousands of pretrained models to perform tasks on different modalities such as
text, vision, and audio. The library was first released in 2018 and has since grown to
support over 100 model architectures. It is compatible with PyTorch, TensorFlow, and JAX,
and is used by researchers and practitioners worldwide to democratize machine learning.
"""

questions = [
    "When was the Hugging Face Transformers library first released?",
    "How many model architectures does it support?",
    "What frameworks is the library compatible with?",
    "Who originally created the Transformers library?",
]

print(f"Context:\n{context.strip()}\n")
print("-" * 70)
for q in questions:
    result = qa_pipeline(question=q, context=context)
    print(f"Q: {q}")
    print(f"A: {result['answer']}  (score: {result['score']:.4f}, "
          f"start: {result['start']}, end: {result['end']})")
    print()

## 3. Low-Level Extractive QA

Let's work directly with the model to understand how span prediction works.

In [ ]:
MODEL_NAME = "distilbert-base-cased-distilled-squad"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
qa_model = AutoModelForQuestionAnswering.from_pretrained(MODEL_NAME).to(device)
qa_model.eval()

def answer_question(question: str, context: str, top_k: int = 1) -> list[dict]:
    """
    Perform extractive QA.

    Returns a list of dicts with 'answer', 'start', 'end', 'score' keys.
    """
    inputs = tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        stride=128,
        return_overflowing_tokens=False,
        return_offsets_mapping=True,
    ).to(device)

    offset_mapping = inputs.pop("offset_mapping").squeeze()

    with torch.no_grad():
        outputs = qa_model(**inputs)

    start_logits = outputs.start_logits.squeeze()
    end_logits = outputs.end_logits.squeeze()

    # Find the best answer span
    start_idx = torch.argmax(start_logits).item()
    end_idx = torch.argmax(end_logits).item()

    # Map token positions back to character positions
    start_char = offset_mapping[start_idx][0].item()
    end_char = offset_mapping[end_idx][1].item()
    answer_text = context[start_char:end_char]

    import torch.nn.functional as F
    score = (F.softmax(start_logits, dim=0)[start_idx] *
             F.softmax(end_logits, dim=0)[end_idx]).item()

    return [{"answer": answer_text, "start": start_char, "end": end_char, "score": score}]


# Test
q = "Who created the Hugging Face Transformers library?"
results = answer_question(q, context)
print(f"Q: {q}")
print(f"A: {results[0]['answer']}  (score: {results[0]['score']:.4f})")

## 4. Evaluating QA with Exact Match and F1

The standard SQuAD metrics are:
- **Exact Match (EM)**: 1 if the predicted answer matches the reference *exactly* (after normalisation), 0 otherwise.
- **F1**: token-level overlap between prediction and reference.

In [ ]:
import re
import string

def normalize_answer(s: str) -> str:
    """Lower text and remove punctuation, articles, and extra whitespace."""
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = ''.join(ch for ch in s if ch not in string.punctuation)
    s = ' '.join(s.split())
    return s


def exact_match(prediction: str, ground_truth: str) -> int:
    return int(normalize_answer(prediction) == normalize_answer(ground_truth))


def f1_score_qa(prediction: str, ground_truth: str) -> float:
    pred_tokens = normalize_answer(prediction).split()
    gt_tokens = normalize_answer(ground_truth).split()
    common = set(pred_tokens) & set(gt_tokens)
    if not common:
        return 0.0
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gt_tokens)
    return 2 * precision * recall / (precision + recall)


# Mini eval dataset: (context, question, reference_answer)
eval_data = [
    (
        "The Apollo 11 mission landed on the Moon on July 20, 1969. "
        "Neil Armstrong was the first human to walk on the lunar surface.",
        "When did Apollo 11 land on the Moon?",
        "July 20, 1969",
    ),
    (
        "The Apollo 11 mission landed on the Moon on July 20, 1969. "
        "Neil Armstrong was the first human to walk on the lunar surface.",
        "Who was the first human to walk on the Moon?",
        "Neil Armstrong",
    ),
    (
        "Python is a high-level, interpreted programming language created by Guido van Rossum "
        "and first released in 1991. It emphasises code readability.",
        "Who created Python?",
        "Guido van Rossum",
    ),
    (
        "Python is a high-level, interpreted programming language created by Guido van Rossum "
        "and first released in 1991. It emphasises code readability.",
        "When was Python first released?",
        "1991",
    ),
]

total_em, total_f1 = 0, 0
for ctx, q, ref in eval_data:
    pred = answer_question(q, ctx)[0]["answer"]
    em = exact_match(pred, ref)
    f1 = f1_score_qa(pred, ref)
    total_em += em
    total_f1 += f1
    print(f"Q  : {q}")
    print(f"Ref: {ref}")
    print(f"Pred: {pred}  | EM={em}  F1={f1:.2f}")
    print()

print(f"Average EM : {total_em / len(eval_data):.2f}")
print(f"Average F1 : {total_f1 / len(eval_data):.2f}")

## 5. Multi-Context QA (Open-Domain)

In a real open-domain QA system, we retrieve relevant passages first and then run extractive QA on each. Here we simulate this with multiple contexts.

In [ ]:
def open_domain_qa(question: str, contexts: list[str]) -> dict:
    """
    Run QA over multiple contexts and return the best answer.

    Parameters
    ----------
    question : str
    contexts : list of candidate paragraphs

    Returns
    -------
    dict with keys: 'answer', 'score', 'context'
    """
    best = {"answer": "", "score": -1.0, "context": ""}
    for ctx in contexts:
        result = qa_pipeline(question=question, context=ctx)
        if result["score"] > best["score"]:
            best = {"answer": result["answer"], "score": result["score"], "context": ctx}
    return best


passages = [
    "The Eiffel Tower is a wrought-iron lattice tower on the Champ de Mars in Paris, France. "
    "It was constructed from 1887 to 1889 as the centrepiece of the 1889 World's Fair.",

    "The Great Wall of China is a series of fortifications built across the historical northern "
    "borders of China to protect against various nomadic groups. Construction began as early as the 7th century BC.",

    "The Amazon River in South America is the largest river by water discharge in the world "
    "and has the largest drainage basin. It flows mainly through Brazil.",
]

question = "When was the Eiffel Tower constructed?"
answer = open_domain_qa(question, passages)

print(f"Question: {question}")
print(f"Answer  : {answer['answer']}  (score: {answer['score']:.4f})")
print(f"Context : {answer['context'][:80]}...")

## 6. Visualising Answer Confidence

In [ ]:
import matplotlib.pyplot as plt

qa_examples = [
    {
        "question": "When did Apollo 11 land on the Moon?",
        "context": "The Apollo 11 mission landed on the Moon on July 20, 1969.",
    },
    {
        "question": "Who created Python?",
        "context": "Python was created by Guido van Rossum and first released in 1991.",
    },
    {
        "question": "What is the capital of Australia?",
        "context": "Australia's capital city is Canberra, not Sydney as many people mistakenly think.",
    },
    {
        "question": "How many planets are in the solar system?",
        "context": "There are eight planets in our solar system: Mercury, Venus, Earth, Mars, "
                   "Jupiter, Saturn, Uranus, and Neptune.",
    },
]

answers = []
scores = []
for ex in qa_examples:
    res = qa_pipeline(**ex)
    answers.append(res["answer"])
    scores.append(res["score"])

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(
    [f'Q{i+1}: {a[:25]}...' if len(a) > 25 else f'Q{i+1}: {a}'
     for i, a in enumerate(answers)],
    scores,
    color="cornflowerblue",
)
ax.set_xlabel("Confidence score")
ax.set_title("Extractive QA — answer confidence")
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

for i, (ex, ans, sc) in enumerate(zip(qa_examples, answers, scores), 1):
    print(f"Q{i}: {ex['question']}")
    print(f"   Answer: {ans}  (score: {sc:.4f})")

## 7. Summary

In this notebook we:
- Performed extractive QA using DistilBERT fine-tuned on SQuAD
- Implemented Exact Match and F1 evaluation metrics
- Built a simple open-domain QA system over multiple passages
- Visualised model confidence across questions

**Extension Ideas**
- Replace keyword search with dense retrieval (DPR, FAISS)
- Use a generative QA model (T5, BART) for free-form answers
- Integrate with a document store for large-scale retrieval-augmented QA

**Next**: `05_sentiment_classification.ipynb` — fine-tuning a transformer for sentiment analysis.